In [1]:
import pandas as pd

In [2]:
def y_labels(num):
    num = float('{:.3g}'.format(num))
    magnitude = 0
    while abs(num) >= 1000:
        magnitude += 1
        num /= 1000.0
    rtn = '{}{}'.format('{:f}'.format(num).rstrip('0').rstrip('.'), ['', 'K', 'M', 'B', 'T'][magnitude])
    return rtn.replace('.00', '').replace('.0', '')

def x_labels(date):
    return date.strftime('%Y')

In [3]:
df = pd.read_csv("monthly_petrcons_tonnes_data.csv")
df['YYYYMM'] = pd.to_datetime(df['YYYYMM'], format="%Y%m")
df['year'] = df['YYYYMM'].dt.year
df['month'] = df['YYYYMM'].dt.month_name().str[:3]
df.head()

,YYYYMM,Diesel,Petrol,All others,year,month
0,1998-04-01,3.193434,0.447685,3.538499,1998,Apr
1,1998-05-01,3.273822,0.489861,3.904566,1998,May
2,1998-06-01,3.192915,0.461987,3.631622,1998,Jun
3,1998-07-01,2.920609,0.451383,3.909489,1998,Jul
4,1998-08-01,2.625254,0.449519,3.792533,1998,Aug


In [49]:
x_labels(df['YYYYMM'][0])

'1998'

In [111]:
line_chart = psc.SimpleLineChart(x_values=df['YYYYMM'],
                                 y_values=[df['Petrol'], df['Diesel'],df['All others']],
                                 y_names=['Petrol', 'Diesel', 'All others'],
                                 # x_max_ticks=30,
                                 y_max_ticks=5,
                                 x_label_format=x_labels,
                                 y_label_format=y_labels, 
                                 width=1200)
line_chart.series['Petrol'].styles = {'stroke': "#DB7D33", 'stroke-width': '3'}
line_chart.series['Diesel'].styles = {'stroke': "#800080", 'stroke-width': '3'}
line_chart.series['All others'].styles = {'stroke': '#2D2D2D', 'stroke-width': '3'}
line_chart.add_y_grid(minor_ticks=0, major_grid_style={'stroke': '#E9E9DE'})
line_chart.add_x_grid(minor_ticks=0, major_grid_style={'stroke': '#E9E9DE'})

# Remove default X tick labels so they don't appear twice
for tick in line_chart.x_axis.tick_texts:
    tick.content = ""
    
for limit, tick in zip(line_chart.x_axis.scale.ticks, line_chart.x_axis.tick_texts):
    line_chart.add_custom_element(psc.Text(x=tick.position.x+20,
                                           y=tick.position.y,
                                           content=str(limit.year),
                                           styles={**tick.styles,"font-size": "12px"}))
#line_chart.add_legend()

In [116]:
def hover_modifier(position, x_value, y_value, series_name, styles):
    text_styles = {'alignment-baseline': 'middle', 'text-anchor': 'middle'}

    params = {
        'styles': text_styles,
        'classes': ['psc-hover-data']
    }

    return [
        psc.Circle(x=position.x, y=position.y, radius=3,
                   classes=['psc-hover-data'], styles=styles),

        psc.Text(x=position.x, y=position.y - 10, content=str(x_value), **params),
        psc.Text(x=position.x, y=position.y - 30, content="{:,.0f}".format(y_value), **params),
        psc.Text(x=position.x, y=position.y - 50, content=series_name, **params)
    ]

line_chart.add_hover_modifier(hover_modifier, radius=5)

# HIDE hover elements by default
line_chart.styles['.psc-hover-data'] = {'display': 'none'}

# SHOW them when hovering the line or marker
line_chart.styles['.psc-series:hover .psc-hover-data'] = {'display': 'block'}


AttributeError: 'SimpleLineChart' object has no attribute 'styles'

In [ ]:
line_chart.save('test.svg')

In [83]:
dates = [dt.date.today() - dt.timedelta(days=i) for i in range(500) if (dt.date.today() + dt.timedelta(days=i)).weekday() == 0][::-1]
actual = [(1 + math.sin(d.timetuple().tm_yday / 183 * math.pi)) * 50000 + 1000 * i + random.randint(-10000, 10000) for i, d in enumerate(dates)]
expected = [a + random.randint(-10000, 10000) for a in actual]
line_chart = psc.SimpleLineChart(x_values=dates, y_values=[actual, expected], y_names=['Actual sales', 'Predicted sales'], x_max_ticks=30, x_label_format=x_labels, y_label_format=y_labels, width=1200)
line_chart.series['Actual sales'].styles = {'stroke': "#DB7D33", 'stroke-width': '3'}
line_chart.series['Predicted sales'].styles = {'stroke': '#2D2D2D', 'stroke-width': '3', 'stroke-dasharray': '4,4'}
line_chart.add_legend(x=700, element_x=200, line_length=35, line_text_gap=20)
line_chart.add_y_grid(minor_ticks=0, major_grid_style={'stroke': '#E9E9DE'})
line_chart.x_axis.tick_lines, line_chart.y_axis.tick_lines = [], []
line_chart.x_axis.axis_line = None
line_chart.y_axis.axis_line.styles['stroke'] = '#E9E9DE'
line_end = line_chart.legend.lines[0].end
act_styles = {'fill': '#FFFFFF', 'stroke': '#DB7D33', 'stroke-width': '3'}
line_chart.add_custom_element(psc.Circle(x=line_end.x, y=line_end.y, radius=4, styles=act_styles))
line_end = line_chart.legend.lines[1].end
pred_styles = {'fill': '#2D2D2D', 'stroke': '#2D2D2D', 'stroke-width': '3'}
line_chart.add_custom_element(psc.Circle(x=line_end.x, y=line_end.y, radius=4, styles=pred_styles))
for limit, tick in zip(line_chart.x_axis.scale.ticks, line_chart.x_axis.tick_texts):
    if tick.content == 'Jan':
        line_chart.add_custom_element(psc.Text(x=tick.position.x, y=tick.position.y + 15, content=str(limit.year), styles=tick.styles))

def hover_modifier(position, x_value, y_value, series_name, styles):
    text_styles = {'alignment-baseline': 'middle', 'text-anchor': 'middle'}
    params = {'styles': text_styles, 'classes': ['psc-hover-data']}
    return [
        psc.Circle(x=position.x, y=position.y, radius=3, classes=['psc-hover-data'], styles=styles),
        psc.Text(x=position.x, y=position.y - 10, content=str(x_value), **params),
        psc.Text(x=position.x, y=position.y - 30, content="{:,.0f}".format(y_value), **params),
        psc.Text(x=position.x, y=position.y - 50, content=series_name, **params)
    ]

line_chart.add_hover_modifier(hover_modifier, radius=5)
line_chart.save('test2.svg')

NameError: name 'dt' is not defined

In [125]:
row['month']

'Nov'

In [6]:
import pygal

# Get unique years and their first occurrence positions
unique_years = []
year_positions = {}
for idx, row in df.iterrows():
    year = row['year']
    if year not in year_positions:
        year_positions[year] = idx
        unique_years.append(year)

# Create x-axis labels - show year only at first occurrence
x_labels = []
for idx, row in df.iterrows():
    if idx in year_positions.values():
        x_labels.append(str(row['year']))
    else:
        x_labels.append('')
guide_positions = list(year_positions.values())

# Create the line chart
line_chart = pygal.Line(
    x_title='Year',
    y_title='Quantity (in Mt)',
    # title='Petrol vs Diesel Prices Over Time',
    show_minor_x_labels=True,
    show_x_labels=True,
    show_y_labels=True,
    show_y_guides=True,  # Show horizontal grid lines
    
    # 💡 ADD THIS LINE 💡
    x_guides=guide_positions, # Show vertical guides only at the start of each year
    
    x_labels_major_every=12,
    y_labels_major_every=2,
    x_labels_major_count=len(unique_years),
    dots_size=2,
    stroke_style={'width': 2.5},
    legend_at_bottom=True,
    truncate_label=-1
)

# Set x-axis labels
line_chart.x_labels = x_labels
line_chart.x_labels_major = x_labels

# Set y-axis labels
line_chart.y_labels = [2, 4, 6, 8, 10]
line_chart.y_labels_major = [2, 4, 6, 8, 10]

# Add data series
line_chart.add('Petrol', df['Petrol'].tolist())
line_chart.add('Diesel', df['Diesel'].tolist())
line_chart.add('All others', df['All others'].tolist())

# Render to file
line_chart.render_to_file('fuel_prices_chart.svg')